In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from matplotlib.animation import FuncAnimation, PillowWriter
%matplotlib inline
from IPython.display import HTML
ss_df=pd.read_csv("solar_system_data.csv")

#Variables
mass_of_sun = 1.989e30 #(kg)
mass_of_earth = 5.98e24 #(kg) 
G= 6.6743e-11  #(Gravitational Constant in m3 kg−1 s−2) 
#Sun is at position (0,0) 
x_sun,y_sun=(0,0) 
earth_aphelion=1.521e11 #(farthest distance from sun in meters)
earth_perihelion=1.471e11 #(closest distance from sun in meters)
eccentricity_earth= 0.0167
semi_major_axis= (1.471e11+1.521e11)/2
semi_minor_axis=semi_major_axis*math.sqrt(1-(eccentricity_earth**2))

#Time Parameters
dt=60*60*24 #1 hour in seconds 
total_days = 365
num_steps = total_days 
#Force calculated in two dimensional plane with x and y components
#Initial Conditions
x_earth = earth_perihelion  # start at perihelion
y_earth = 0
vx = 0
vy = math.sqrt(G * mass_of_sun * (2/earth_perihelion - 1/semi_major_axis))


#Trajectory Array Setup
x_history = []
y_history = []

for i in range(num_steps):
    
    r_earth_sun = math.sqrt(x_earth**2 + y_earth**2)
    # Gravitational force components
    F = G * mass_of_sun * mass_of_earth / r_earth_sun**2
    Fx = -F * (x_earth / r_earth_sun)
    Fy = -F * (y_earth / r_earth_sun)

    #Acceleration
    ax = Fx / mass_of_earth
    ay = Fy / mass_of_earth

    # Update velocity
    vx += ax * dt
    vy += ay * dt
    
    # Update position
    x_earth += vx * dt
    y_earth += vy * dt
    
    #Appending
    x_history.append(x_earth)
    y_history.append(y_earth)

fig, ax = plt.subplots(figsize=(6,6))

limit = semi_major_axis * 1.1
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_aspect('equal') 
ax.set_facecolor('black') 

#The Sun
ax.plot(0, 0, 'yo', markersize=10, label='Sun')

#
line, = ax.plot([], [], 'b-', lw=1, alpha=0.5) # Trail
point, = ax.plot([], [], 'bo', markersize=5)   # Earth


def update(frame):
    # Update the line to show history up to the current frame
    line.set_data(x_history[:frame], y_history[:frame])
    # Update the point to show current position
    point.set_data([x_history[frame]], [y_history[frame]])
    return line, point
ani = FuncAnimation(fig, update, frames=len(x_history), interval=20, blit=True)
plt.legend()
plt.title("Earth Orbit Simulation")
plt.close()
HTML(ani.to_jshtml())
writer = PillowWriter(fps = 30) 
ani.save("Earth_Orbit.gif", writer=writer,)